# Loading 

In [103]:
import pandas as pd
import numpy as np

In [104]:

path = r"D:\桌面\研一\DATA70202 Applying Data Science 202526 2nd Semester\AR_2025_Globais_by district and Parish.xlsx"

In [105]:
df_g = pd.read_excel(path, sheet_name='AR_2025_Global', header=3)
df_dist = pd.read_excel(path, sheet_name='AR_2025_Distrito_Reg.Autónoma', header=3)
df_muni = pd.read_excel(path, sheet_name='AR_2025_Concelho', header=3)
df_par = pd.read_excel(path, sheet_name='AR_2025_Freguesia', header=3)
df_na = pd.read_excel(path, sheet_name='AR_2025_País', header=3)
df_abd = pd.read_excel(path, sheet_name='AR_2025_Consulado', header=3)

In [106]:
df_muni.head()

,código,nome do território,ano,total de concelhos,total de freguesias,total de consulados,inscritos,votantes,% votantes,brancos,...,PPM,% votos.16,PS,% votos.17,PTP,% votos.18,R.I.R.,% votos.19,VP,% votos.20
0,500000,Território Nacional,2025.0,308.0,3092.0,NaN,9265493.0,5965446.0,64.38,85966.0,...,5296.0,0.09,1394501.0,23.38,421.0,0.01,13357.0,0.22,10998.0,0.18
1,010100,Águeda,2025.0,1.0,11.0,NaN,41669.0,26061.0,62.54,483.0,...,21.0,0.08,5677.0,21.78,NaN,NaN,38.0,0.15,36.0,0.14
2,010200,Albergaria-a-Velha,2025.0,1.0,6.0,NaN,22407.0,14355.0,64.06,198.0,...,5.0,0.03,2520.0,17.55,NaN,NaN,35.0,0.24,35.0,0.24
3,010300,Anadia,2025.0,1.0,10.0,NaN,25449.0,15526.0,61.01,303.0,...,7.0,0.05,2777.0,17.89,NaN,NaN,36.0,0.23,22.0,0.14
4,010400,Arouca,2025.0,1.0,16.0,NaN,19696.0,12971.0,65.86,209.0,...,28.0,0.22,2145.0,16.54,NaN,NaN,36.0,0.28,16.0,0.12


In [107]:
df_dhmuni = df_muni.copy()
df_dhmuni.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 55 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   código               310 non-null    object 
 1   nome do território   310 non-null    object 
 2   ano                  309 non-null    float64
 3   total de concelhos   309 non-null    float64
 4   total de freguesias  309 non-null    float64
 5   total de consulados  0 non-null      float64
 6   inscritos            309 non-null    float64
 7   votantes             309 non-null    float64
 8   % votantes           309 non-null    float64
 9   brancos              309 non-null    float64
 10  % brancos            309 non-null    float64
 11  nulos                309 non-null    float64
 12  % nulos              309 non-null    float64
 13  ADN                  309 non-null    float64
 14  % votos              309 non-null    float64
 15  B.E.                 309 non-null    flo

## data clean

In [108]:
id_cols = ['código', 'nome do território', 'inscritos']

party_cols = [
    'ADN','B.E.','CH','E','IL','JPP','L','MPT','NC','ND',
    'PAN','PCP-PEV','PCTP/MRPP','PLS',
    'PPD/PSD.CDS-PP','PPD/PSD.CDS-PP.PPM','PPM',
    'PS','PTP','R.I.R.','VP'
]

df_dhmuni = df_muni.melt(
    id_vars=id_cols,
    value_vars=party_cols,
    var_name='party',
    value_name='votes'
)

df_dhmuni = df_dhmuni.dropna(subset=['votes'])
df_dhmuni = df_dhmuni[df_dhmuni['votes'] > 0]

# 删除全国汇总行
df_dhmuni = df_dhmuni[df_dhmuni['nome do território'] != 'Território Nacional']

df_dhmuni = df_dhmuni.reset_index(drop=True)

In [109]:
df_dhmuni

,código,nome do território,inscritos,party,votes
0,010100,Águeda,41669.0,ADN,475.0
1,010200,Albergaria-a-Velha,22407.0,ADN,273.0
2,010300,Anadia,25449.0,ADN,311.0
3,010400,Arouca,19696.0,ADN,285.0
4,010500,Aveiro,70189.0,ADN,677.0
...,...,...,...,...,...
4487,182000,Tarouca,7180.0,VP,6.0
4488,182100,Tondela,24181.0,VP,11.0
4489,182200,Vila Nova de Paiva,5672.0,VP,3.0
4490,182300,Viseu,92594.0,VP,78.0


In [110]:
df_lp = df_muni[df_muni['código'].astype(str).str.startswith(('11','13'))]
df_lp[['código','nome do território']]

,código,nome do território
148,110100,Alenquer
149,110200,Arruda dos Vinhos
150,110300,Azambuja
151,110400,Cadaval
152,110500,Cascais
153,110600,Lisboa
154,110700,Loures
155,110800,Lourinhã
156,110900,Mafra
157,111000,Oeiras


# Group

In [111]:
# --- 1. 提取原始区划前缀 (District Prefix) ---
df_dhmuni['district_prefix'] = df_dhmuni['código'].astype(str).str.zfill(6).str[:2]

# --- 2. 准备行政区名称映射字典 ---
dist_map_dict = df_dist.dropna(subset=['código', 'nome do território']).copy()
dist_map_dict['prefix'] = dist_map_dict['código'].astype(str).str.zfill(6).str[:2]
dist_map_dict = dist_map_dict.set_index('prefix')['nome do território'].to_dict()

# --- 3. 初始化新选区列 ---
# 此时 new_district_code 暂时存放前缀，稍后会被自动生成的序列号覆盖
df_dhmuni['new_district_code'] = df_dhmuni['district_prefix']
df_dhmuni['new_district_name'] = df_dhmuni['district_prefix'].map(dist_map_dict)

# 特殊处理自治区名称（确保 3xxx/4xxx 统一）
df_dhmuni.loc[df_dhmuni['district_prefix'].str.startswith('3'), 'new_district_name'] = "Região Autónoma da Madeira"
df_dhmuni.loc[df_dhmuni['district_prefix'].str.startswith('4'), 'new_district_name'] = "Região Autónoma dos Açores"

# --- 4. 定义子选区划分逻辑 (里斯本 & 波尔图) ---
lisbon_map = {
    'Loures':1,'Odivelas':1,'Vila Franca de Xira':1,'Torres Vedras':1,'Mafra':1,
    'Alenquer':1,'Lourinhã':1,'Azambuja':1,'Cadaval':1,'Arruda dos Vinhos':1,
    'Sobral de Monte Agraço':1,
    'Sintra':2,'Cascais':2,'Amadora':2,
    'Lisboa':3,'Oeiras':3
}

porto_map = {
    'Vila Nova de Gaia':1,'Porto':1,'Matosinhos':1,'Vila do Conde':1,'Póvoa de Varzim':1,'Trofa':1,
    'Gondomar':2,'Maia':2,'Valongo':2,'Paredes':2,'Penafiel':2,'Santo Tirso':2,
    'Felgueiras':2,'Amarante':2,'Paços de Ferreira':2,'Marco de Canaveses':2,
    'Lousada':2,'Baião':2
}

def assign_sub_district_name(row, mapping):
    """仅处理名称划分，不再在此处处理代码"""
    val = mapping.get(row['nome do território'])
    if pd.notna(val):
        return f"{row['new_district_name']} {int(val)}"
    return row['new_district_name']

# 执行里斯本和波尔图名称划分
mask_l = df_dhmuni['district_prefix'] == '11'
df_dhmuni.loc[mask_l, 'new_district_name'] = df_dhmuni[mask_l].apply(lambda x: assign_sub_district_name(x, lisbon_map), axis=1)

mask_p = df_dhmuni['district_prefix'] == '13'
df_dhmuni.loc[mask_p, 'new_district_name'] = df_dhmuni[mask_p].apply(lambda x: assign_sub_district_name(x, porto_map), axis=1)

# --- 5. 执行 A/B/C 组聚合逻辑 (仅更新名称) ---
import numpy as np

conditions = [
    df_dhmuni['district_prefix'].isin(['04', '17']),       # Group A
    df_dhmuni['district_prefix'].isin(['02', '07', '12']), # Group B
    df_dhmuni['district_prefix'].isin(['05', '09'])        # Group C
]

name_choices = ['Trás-os-Montes', 'Alentejo', 'Beira Baixa']
df_dhmuni['new_district_name'] = np.select(conditions, name_choices, default=df_dhmuni['new_district_name'])

# --- 6. 核心修改：按名称自动生成序列号 (1, 2, 3...) 作为 new_district_code ---

# 获取所有唯一的新选区名称并排序（确保 1, 2, 3 的顺序是可预测的，通常按字母表）
unique_names = sorted(df_dhmuni['new_district_name'].unique())

# 创建映射字典：{名称: 序列号}
name_to_id_map = {name: i + 1 for i, name in enumerate(unique_names)}

# 将 new_district_code 覆盖为自动生成的数字 ID
df_dhmuni['new_district_code'] = df_dhmuni['new_district_name'].map(name_to_id_map)

# 此时 df_dhmuni 已经完全恢复，且 new_district_code 为 1, 2, 3...

# D’Hondt Method

In [112]:
# --- 1. D'Hondt 算法定义 ---
def dhondt(votes, seats):
    if seats <= 0: return [0] * len(votes)
    votes = [float(v) for v in votes]
    n = len(votes)
    allocated = [0] * n
    for _ in range(int(seats)):
        # 计算当前每一轮的商
        quotients = [votes[i] / (allocated[i] + 1) for i in range(n)]
        # 找到最大商的索引
        winner = quotients.index(max(quotients))
        allocated[winner] += 1
    return allocated

# --- 2. 准备选区席位基础表 ---
# 提取唯一的选区选民信息
dist_info = df_dhmuni[['new_district_code', 'new_district_name', 'nome do território', 'inscritos']].drop_duplicates()
dist_registered = dist_info.groupby(['new_district_code', 'new_district_name'])['inscritos'].sum().reset_index()

# 分配 226 个席位到各选区
total_seats = 226
voters_list = dist_registered['inscritos'].tolist()
dist_registered['district_total_seats'] = dhondt(voters_list, total_seats)

print("选区席位分配完成。")

# --- 3. 统计各选区内各政党的总票数 ---
party_votes = df_dhmuni.groupby(['new_district_code', 'new_district_name', 'party'])['votes'].sum().reset_index()

# --- 4. 核心逻辑：在选区内根据政党得票分配席位 ---
all_results = []

for _, row in dist_registered.iterrows():
    d_code = row['new_district_code']
    d_name = row['new_district_name']
    d_seats = row['district_total_seats']
    
    # 筛选该选区的所有政党及其票数
    district_data = party_votes[party_votes['new_district_code'] == d_code].copy()
    
    if d_seats > 0:
        # 按照 D'Hondt 分配
        votes_list = district_data['votes'].tolist()
        allocated_seats = dhondt(votes_list, d_seats)
        district_data['allocated_seats'] = allocated_seats
    else:
        district_data['allocated_seats'] = 0
        
    all_results.append(district_data)

# 合并原始计算结果（包含 0 席位政党）
df_seats_final = pd.concat(all_results).reset_index(drop=True)

# --- 5. 清洗并创建 df_per_party ---
# 保留：政党名(party), 获票数(votes), 席位数(allocated_seats), 新选区名(new_district_name)
df_per_party = df_seats_final[['party', 'votes', 'allocated_seats', 'new_district_name']].copy()

# 重命名列名以便更直观（可选）
df_per_party.columns = ['Party', 'Votes', 'Seats', 'District_Name']

# 排序：先按选区名升序，再按席位数降序，最后按得票数降序
df_per_party = df_per_party.sort_values(
    by=['District_Name', 'Seats', 'Votes'], 
    ascending=[True, False, False]
).reset_index(drop=True)


选区席位分配完成。


In [113]:
# 1. 重新构建 df_per_party，确保包含所有必要字段
# 使用汇总后的 party_votes 和之前计算的 df_seats_final 合并
df_per_party = df_seats_final[['new_district_name', 'party', 'votes', 'allocated_seats']].copy()
df_per_party.columns = ['District_Name', 'Party', 'Votes', 'Seats']

# 2. 提取席位透视表 (选区为行，政党为列)
df_seats_pivot = df_per_party.pivot_table(
    index='District_Name', 
    columns='Party', 
    values='Seats', 
    aggfunc='sum'
).fillna(0)

# 3. 提取票数透视表 (选区为行，政党为列)
df_votes_pivot = df_per_party.pivot_table(
    index='District_Name', 
    columns='Party', 
    values='Votes', 
    aggfunc='sum'
).fillna(0)

# 4. 构造交错索引 (Interleaving)
# 为席位行创建索引 (选区名, 0)
df_seats_pivot.index = pd.MultiIndex.from_tuples([(n, 0) for n in df_seats_pivot.index])
# 为票数行创建索引 (选区名, 1)
df_votes_pivot.index = pd.MultiIndex.from_tuples([(n, 1) for n in df_votes_pivot.index])

# 5. 合并并排序，实现“一行席位，一行票数”
df_matrix_final = pd.concat([df_seats_pivot, df_votes_pivot]).sort_index()

# 6. 最终格式化索引名字
new_indices = []
for dist_name, type_flag in df_matrix_final.index:
    if type_flag == 0:
        new_indices.append(dist_name)          # 第一行显示选区名 (代表Seats)
    else:
        new_indices.append("Votes") # 第二行显示票数标识

df_matrix_final.index = new_indices


# Output

In [114]:
dist_registered

,new_district_code,new_district_name,inscritos,district_total_seats
0,1,Alentejo,343472.0,8
1,2,Aveiro,642044.0,16
2,3,Beira Baixa,302080.0,7
3,4,Braga,782367.0,20
4,5,Coimbra,370773.0,9
5,6,Faro,383944.0,9
6,7,Leiria,411995.0,10
7,8,Lisboa 1,663501.0,16
8,9,Lisboa 2,643380.0,16
9,10,Lisboa 3,606064.0,15


In [115]:
# 确定目标文件名
target_file = '2025_D‘Hondt_Result.xlsx'

# 准备要导出的数据：选区席位分配表
# 我们通常只需要选区名、选民数和分配到的席位
df_seats_to_export = dist_registered[['new_district_name', 'inscritos', 'district_total_seats']].copy()

# 按照席位数从高到低排序
df_seats_to_export = df_seats_to_export.sort_values(by='district_total_seats', ascending=False)

try:
    # 使用 mode='a' 可以在现有文件中追加 Sheet
    # 如果文件不存在，会自动进入 except 模块新建文件
    with pd.ExcelWriter(target_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_seats_to_export.to_excel(writer, sheet_name='2025_Seats_Allocation', index=False)
    print(f"成功！选区席位分配表已加入 '{target_file}' 的 '2025_Seats_Allocation' 工作表。")

except FileNotFoundError:
    # 如果文件尚未创建，则直接新建
    df_seats_to_export.to_excel(target_file, sheet_name='2025_Seats_Allocation', index=False)
    print(f"已新建文件 '{target_file}' 并保存选区席位分配表。")

成功！选区席位分配表已加入 '2025_D‘Hondt_Result.xlsx' 的 '2025_Seats_Allocation' 工作表。


In [116]:
# 7. 导出结果
file_path = '2025_D‘Hondt_Result.xlsx'
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    # 保存矩阵报表
    df_matrix_final.to_excel(writer, sheet_name='2025_per_party')
print(f"成功导出！请查看 '{file_path}' 中的 '2025_per_party' 工作表。")

# 打印预览
print(df_matrix_final.head(4))

成功导出！请查看 '2025_D‘Hondt_Result.xlsx' 中的 '2025_per_party' 工作表。
Party        ADN    B.E.       CH      E       IL  JPP        L  MPT     NC  \
Alentejo     0.0     0.0      2.0    0.0      0.0  0.0      0.0  0.0    0.0   
Votes     2315.0  3585.0  58747.0  222.0   4961.0  0.0   4829.0  0.0  106.0   
Aveiro       0.0     0.0      4.0    0.0      1.0  0.0      0.0  0.0    0.0   
Votes     6974.0  7015.0  85760.0  528.0  23444.0  0.0  12805.0  0.0  299.0   

Party        ND  ...  PCP-PEV  PCTP/MRPP  PLS  PPD/PSD.CDS-PP  \
Alentejo    0.0  ...      1.0        0.0  0.0             2.0   
Votes      97.0  ...  21614.0      718.0  0.0         51935.0   
Aveiro      0.0  ...      0.0        0.0  0.0             7.0   
Votes     472.0  ...   4923.0     1005.0  0.0        163617.0   

Party     PPD/PSD.CDS-PP.PPM    PPM       PS  PTP  R.I.R.     VP  
Alentejo                 0.0    0.0      3.0  0.0     0.0    0.0  
Votes                    0.0  203.0  59277.0  0.0   352.0  307.0  
Aveiro          